# Glass Ball Impact Video (20 frames)

Build a 20-frame animation of a glass ball falling and shattering, then export an MP4.

| Mode | When to use |
|------|-------------|
| **`local`** (default) | No API quota needed — renders frames with Pillow |
| **`gemini`** | Requires paid Gemini API billing for image generation |
| **`auto`** | Try Gemini once; fall back to `local` on quota/404 errors |

**Gemini note:** Free tier has **zero quota** for image models (`429 RESOURCE_EXHAUSTED`). Enable billing at [ai.google.dev](https://ai.google.dev) to use `gemini` mode.

In [ ]:
%pip install -q pillow imageio imageio-ffmpeg numpy
%pip install -q google-genai  # only needed for gemini/auto modes

In [ ]:
import math
import os
import random
import time
from io import BytesIO
from pathlib import Path

import imageio.v2 as imageio
import numpy as np
from IPython.display import Image as IPImage, Video, display
from PIL import Image, ImageDraw

# "local" | "gemini" | "auto"
GENERATION_MODE = "local"

NOTEBOOK_DIR = Path.cwd()
FRAMES_DIR = NOTEBOOK_DIR / "frames_glass_ball"
VIDEO_PATH = NOTEBOOK_DIR / "glass_ball_impact.mp4"
NUM_FRAMES = 20
FPS = 8
CANVAS_SIZE = (640, 480)

# Gemini image model (requires paid tier for image output)
GEMINI_IMAGE_MODEL = "gemini-2.5-flash-image"
GEMINI_RETRY_SECONDS = 45  # wait on 429 before retry (or fallback in auto mode)


def load_api_key():
    try:
        from google.colab import userdata
        return userdata.get("GEMINI_API_KEY")
    except ImportError:
        pass
    env_file = Path.home() / "Workspace" / ".env"
    if env_file.exists():
        for line in env_file.read_text().splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip())
    return os.environ.get("GEMINI_API_KEY")


client = None
if GENERATION_MODE in ("gemini", "auto"):
    key = load_api_key()
    if not key:
        if GENERATION_MODE == "gemini":
            raise ValueError("GEMINI_API_KEY required for gemini mode")
        print("No GEMINI_API_KEY — auto mode will use local rendering")
        GENERATION_MODE = "local"
    else:
        from google import genai
        from google.genai import types
        client = genai.Client(api_key=key)
        print(f"Gemini client ready (mode={GENERATION_MODE})")
else:
    print(f"Using local rendering (mode={GENERATION_MODE})")



In [ ]:
def _draw_background(draw: ImageDraw.ImageDraw, w: int, h: int, ground_y: int):
    draw.rectangle([0, 0, w, h], fill=(28, 32, 40))
    draw.rectangle([0, ground_y, w, h], fill=(90, 92, 98))
    draw.line([(0, ground_y), (w, ground_y)], fill=(120, 122, 128), width=2)


def _draw_glass_ball(draw, cx, cy, rx, ry, alpha=255):
    bbox = [cx - rx, cy - ry, cx + rx, cy + ry]
    draw.ellipse(bbox, fill=(180, 220, 255, alpha))
    draw.ellipse([cx - rx * 0.55, cy - ry * 0.55, cx - rx * 0.1, cy - ry * 0.15], fill=(255, 255, 255, min(180, alpha)))
    draw.arc(bbox, 200, 340, fill=(120, 180, 230, alpha), width=3)


def _draw_cracks(draw, cx, cy, r, intensity):
    for i in range(intensity):
        angle = (i / max(intensity, 1)) * math.pi * 2
        x2 = cx + math.cos(angle) * r * 0.9
        y2 = cy + math.sin(angle) * r * 0.9
        draw.line([(cx, cy), (x2, y2)], fill=(200, 230, 255), width=2)


def _draw_shards(draw, cx, cy, frame_idx, seed=42):
    rng = random.Random(seed + frame_idx)
    count = 6 + (frame_idx - 14) * 4
    for _ in range(count):
        angle = rng.uniform(0, math.pi * 2)
        dist = rng.uniform(20, 60 + frame_idx * 4)
        sx = cx + math.cos(angle) * dist
        sy = cy + math.sin(angle) * dist * 0.35
        size = rng.uniform(8, 22)
        pts = [
            (sx, sy),
            (sx + size, sy + size * 0.3),
            (sx + size * 0.4, sy + size),
            (sx - size * 0.2, sy + size * 0.6),
        ]
        draw.polygon(pts, fill=(170, 210, 245, 200), outline=(220, 240, 255))


def generate_frame_local(frame_index: int) -> Image.Image:
    """Render one animation frame without any API call."""
    w, h = CANVAS_SIZE
    ground_y = int(h * 0.82)
    ball_r = 36
    img = Image.new("RGBA", (w, h), (0, 0, 0, 255))
    draw = ImageDraw.Draw(img, "RGBA")
    _draw_background(draw, w, h, ground_y)

    cx = w // 2
    # Normalized timeline 0..1 across 20 frames
    t = (frame_index - 1) / (NUM_FRAMES - 1)

    if frame_index <= 11:
        # Fall: ease-in quadratic
        fall_t = ((frame_index - 1) / 10) ** 2
        cy = int(90 + (ground_y - ball_r - 90) * fall_t)
        _draw_glass_ball(draw, cx, cy, ball_r, ball_r)
    elif frame_index == 12:
        cy = ground_y - ball_r + 6
        _draw_glass_ball(draw, cx, cy, ball_r + 6, ball_r - 10)
    elif frame_index == 13:
        cy = ground_y - ball_r + 10
        _draw_glass_ball(draw, cx, cy, ball_r + 8, ball_r - 14)
        _draw_cracks(draw, cx, cy - 4, ball_r, 6)
    elif frame_index == 14:
        cy = ground_y - ball_r + 12
        _draw_glass_ball(draw, cx, cy, ball_r + 4, ball_r - 18)
        _draw_cracks(draw, cx, cy - 6, ball_r, 10)
        _draw_shards(draw, cx, cy, frame_index)
    else:
        impact_y = ground_y - 10
        _draw_shards(draw, cx, impact_y, frame_index)
        if frame_index < 20:
            _draw_glass_ball(draw, cx, impact_y - 8, max(8, ball_r - (frame_index - 13) * 5), max(6, ball_r - (frame_index - 12) * 6), alpha=120)
        # dust
        for i in range(8 + frame_index):
            dx = cx + random.randint(-80, 80)
            dy = ground_y + random.randint(-8, 4)
            draw.ellipse([dx, dy, dx + 3, dy + 3], fill=(200, 200, 200, 80))

    return img.convert("RGB")


# Quick preview of local frame 1
preview = generate_frame_local(1)
preview



In [ ]:
BASE_STYLE = (
    "Photorealistic 3D render, side profile, fixed camera, dark background, gray concrete floor. "
    "Clear glass sphere with reflections. Same composition every frame; only motion changes. "
)

FRAME_STAGES = [
    "Glass ball stationary at the top, far above ground.",
    "Ball just started falling.",
    "Ball in upper quarter of frame.",
    "Ball in upper third.",
    "Ball in upper-middle area.",
    "Ball in middle of frame.",
    "Ball in lower-middle area.",
    "Ball in lower third.",
    "Ball in lower quarter.",
    "Ball very close to ground.",
    "Ball millimeters above ground before impact.",
    "First contact, slight flattening at bottom.",
    "Impact compression, hairline cracks.",
    "Cracks spread, small shards separate.",
    "Ball shatters, fragments fly outward.",
    "More fragments scatter on ground.",
    "Shards spread with motion blur.",
    "Fragments slowing and settling.",
    "Most shards on floor, few tumbling.",
    "Final: scattered shards, no intact sphere.",
]

FRAME_PROMPTS = [
    f"{BASE_STYLE} Frame {i + 1}/{NUM_FRAMES}: {s}" for i, s in enumerate(FRAME_STAGES)
]


def extract_image_from_response(response) -> Image.Image:
    if not response.candidates:
        raise RuntimeError("No candidates in response")
    for part in response.candidates[0].content.parts:
        if part.inline_data and part.inline_data.data:
            return Image.open(BytesIO(part.inline_data.data))
    raise RuntimeError("No image in response")


def _is_quota_error(exc: Exception) -> bool:
    return "429" in str(exc) or "RESOURCE_EXHAUSTED" in str(exc) or "quota" in str(exc).lower()


def generate_frame_gemini(prompt: str, frame_index: int) -> Image.Image:
    from google.genai import types

    for attempt in range(2):
        try:
            response = client.models.generate_content(
                model=GEMINI_IMAGE_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(response_modalities=["TEXT", "IMAGE"]),
            )
            return extract_image_from_response(response)
        except Exception as e:
            if _is_quota_error(e) and attempt == 0:
                print(f"  Quota hit — waiting {GEMINI_RETRY_SECONDS}s before retry...")
                time.sleep(GEMINI_RETRY_SECONDS)
                continue
            raise



In [ ]:
def generate_frame(frame_index: int, prompt: str | None = None) -> Image.Image:
    mode = GENERATION_MODE
    if mode == "local":
        return generate_frame_local(frame_index)
    if mode == "gemini":
        return generate_frame_gemini(prompt, frame_index)
    # auto
    try:
        return generate_frame_gemini(prompt, frame_index)
    except Exception as e:
        print(f"  Gemini failed ({type(e).__name__}); using local renderer")
        return generate_frame_local(frame_index)


use_gemini = GENERATION_MODE in ("gemini", "auto")
FRAMES_DIR.mkdir(parents=True, exist_ok=True)
frames = []
gemini_failed_globally = False

for i in range(1, NUM_FRAMES + 1):
    path = FRAMES_DIR / f"frame_{i:02d}.png"
    prompt = FRAME_PROMPTS[i - 1] if use_gemini else None

    if path.exists():
        img = Image.open(path).convert("RGB")
        print(f"Loaded cached {path.name}")
    else:
        print(f"Generating frame {i}/{NUM_FRAMES}...")
        if use_gemini and not gemini_failed_globally:
            try:
                img = generate_frame(i, prompt)
                print(f"  (gemini: {GEMINI_IMAGE_MODEL})")
            except Exception as e:
                if GENERATION_MODE == "auto" and (_is_quota_error(e) or "404" in str(e)):
                    print(f"  Switching all remaining frames to local mode: {e}")
                    gemini_failed_globally = True
                    img = generate_frame_local(i)
                    print("  (local fallback)")
                else:
                    raise
        else:
            img = generate_frame_local(i)
            print("  (local)")
        img.save(path)

    frames.append(img)

print(f"\n{len(frames)} frames ready in {FRAMES_DIR}")



In [ ]:
target_size = frames[0].size
video_frames = [np.array(f.resize(target_size, Image.Resampling.LANCZOS)) for f in frames]

imageio.mimsave(VIDEO_PATH, video_frames, fps=FPS, codec="libx264", quality=8)
print(f"Video saved: {VIDEO_PATH} ({len(video_frames)} frames @ {FPS} fps)")
display(Video(str(VIDEO_PATH), embed=True, width=640))



In [ ]:
for idx in [0, 4, 9, 12, 15, 19]:
    path = FRAMES_DIR / f"frame_{idx + 1:02d}.png"
    if path.exists():
        print(f"Frame {idx + 1}")
        display(IPImage(filename=str(path), width=320))

